In [ ]:
try:
    import onnxruntime as ort
    print('onnxruntime already available:', ort.__version__)
except ImportError:
    from pathlib import Path
    import subprocess, sys
    _whl = next(Path('/kaggle/input').rglob('onnxruntime-*.whl'), None)
    if _whl is None:
        raise RuntimeError(
            "No onnxruntime wheel found under /kaggle/input. "
            "Attach a dataset that contains a wheel (e.g. the perch-onnx dataset)."
        )
    print(f'Installing onnxruntime from {_whl.name}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           '--no-deps', '--no-index', str(_whl)])
    import onnxruntime as ort
    print('onnxruntime installed:', ort.__version__)


In [ ]:
from pathlib import Path
import random

BASE = Path('/kaggle/input/birdclef-2026')
assert BASE.exists(), 'Attach the birdclef-2026 competition data'

_sed_dir = None
for _p in [
    '/kaggle/input/birdclef2026-sed-b0-jft',
    '/kaggle/input/datasets/smitapriyadarshani/birdclef2026-sed-b0-jft',
]:
    if Path(_p).exists() and list(Path(_p).glob('*.onnx')):
        _sed_dir = _p
        break
assert _sed_dir, "SED models not found - attach 'birdclef2026-sed-b0-jft' dataset"

SED_PATHS = sorted([p for p in Path(_sed_dir).glob('sed_b0_jft_fold*.onnx')
                    if any(f'fold{i}' in p.name for i in [0, 1])])
assert SED_PATHS, f'No fold0/fold1 ONNX in {_sed_dir}'
print(f'Teacher SED models ({len(SED_PATHS)}):')
for p in SED_PATHS:
    print(f'  {p.name}  ({p.stat().st_size / 1e6:.1f} MB)')

soundscapes = sorted((BASE / 'train_soundscapes').glob('*.ogg'))
print(f'\nTotal train_soundscapes: {len(soundscapes)}')

SEED = 42
MAX_FILES = 2000
random.Random(SEED).shuffle(soundscapes)
files = soundscapes[:MAX_FILES]
print(f'Will pseudo-label {len(files)} soundscape files')


In [ ]:
import numpy as np
import torch, torchaudio

SR = 32000
CLIP_SEC = 5.0
WIN_SAMPLES = int(SR * CLIP_SEC)
FILE_SEC = 60
N_WIN = 12
N_FFT, HOP, N_MELS = 2048, 512, 128
FMIN, FMAX = 50, 14000

_mel_t = torchaudio.transforms.MelSpectrogram(
    sample_rate=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS,
    f_min=FMIN, f_max=FMAX, power=2.0)
_db_t = torchaudio.transforms.AmplitudeToDB(top_db=80)

_IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def file_to_mel_3ch_batch(wav60):
    chunks = np.zeros((N_WIN, WIN_SAMPLES), dtype=np.float32)
    for wi in range(N_WIN):
        s = wi * WIN_SAMPLES
        chunks[wi] = wav60[s:s + WIN_SAMPLES]
    x = torch.from_numpy(chunks)
    with torch.no_grad():
        m = _db_t(_mel_t(x))
        m = (m - m.mean(dim=(-2, -1), keepdim=True)) / \
            (m.std(dim=(-2, -1), keepdim=True) + 1e-6)
    m = m.unsqueeze(1).expand(-1, 3, -1, -1).contiguous()
    m = (m - _IMAGENET_MEAN) / _IMAGENET_STD
    return m.numpy().astype(np.float32)

print('Mel-spec config: SR={}, n_mels={}, fmin={}, fmax={}, n_win={}'.format(
    SR, N_MELS, FMIN, FMAX, N_WIN))


In [ ]:
import pandas as pd
import onnxruntime as ort

so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
so.intra_op_num_threads = 4
sed_sessions = [
    ort.InferenceSession(str(p), sess_options=so, providers=['CPUExecutionProvider'])
    for p in SED_PATHS
]
print(f'Loaded {len(sed_sessions)} SED sessions')

sample_sub = pd.read_csv(BASE / 'sample_submission.csv')
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES = len(PRIMARY_LABELS)
print(f'N_CLASSES = {N_CLASSES}')


In [ ]:
import time, gc, soundfile as sf

OUT_PARQUET = Path('/kaggle/working/soundscape_pseudo_labels.parquet')

rows = []
t0 = time.time()
for fi, fp in enumerate(files):
    try:
        wav, _sr = sf.read(str(fp), dtype='float32', always_2d=False)
        if wav.ndim == 2: wav = wav.mean(axis=1)
        need = FILE_SEC * SR
        if len(wav) < need: wav = np.pad(wav, (0, need - len(wav)))
        else: wav = wav[:need]
    except Exception as e:
        print(f'skip {fp.name}: {e}')
        continue

    mel_batch = file_to_mel_3ch_batch(wav)

    per_model = []
    for sess in sed_sessions:
        iname = sess.get_inputs()[0].name
        p = sess.run(None, {iname: mel_batch})[0]
        per_model.append(p)
    avg_pred = np.mean(per_model, axis=0).astype(np.float32)

    for wi in range(N_WIN):
        rows.append({
            'filename': fp.name,
            'start_sec': int(wi * 5),
            'labels': avg_pred[wi].tolist(),
        })

    if (fi + 1) % 50 == 0:
        elapsed = time.time() - t0
        eta_min = (len(files) - fi - 1) * elapsed / (fi + 1) / 60
        print(f'  {fi+1}/{len(files)} files  ({elapsed:.0f}s, ETA {eta_min:.0f} min)')
    if (fi + 1) % 200 == 0:
        gc.collect()

pdf = pd.DataFrame(rows)
pdf.to_parquet(OUT_PARQUET, index=False)
print(f'\nWrote {OUT_PARQUET}  rows={len(pdf)}  size={OUT_PARQUET.stat().st_size / 1e6:.1f} MB')


In [ ]:
all_labels = np.array(pdf['labels'].tolist(), dtype=np.float32)
print(f'Shape: {all_labels.shape}  (windows x classes)')
print(f'Mean prob:           {all_labels.mean():.4f}')
print(f'Median prob:         {np.median(all_labels):.4f}')
print(f'Max prob:            {all_labels.max():.4f}')
print(f'Per-class mean: min={all_labels.mean(0).min():.4f}, max={all_labels.mean(0).max():.4f}')
print(f'Classes with mean prob > 0.01: {(all_labels.mean(0) > 0.01).sum()}/{N_CLASSES}')

max_per_window = all_labels.max(axis=1)
for thr in [0.1, 0.2, 0.3, 0.5, 0.7, 0.9]:
    print(f'  windows with max-prob > {thr}: {(max_per_window > thr).sum()}/{len(max_per_window)} '
          f'({100 * (max_per_window > thr).mean():.1f}%)')

top_idx = np.argsort(-all_labels.mean(0))[:10]
print('\nTop 10 most-predicted classes (mean prob):')
for i in top_idx:
    print(f'  {PRIMARY_LABELS[i]:<25} {all_labels.mean(0)[i]:.4f}')
